In [0]:
from pyspark.sql import functions as F

CATALOG = "healthcare_medallion_dbw"

billing_bronze = spark.table(
    f"{CATALOG}.bronze.billing"
)

billing_bronze.printSchema()

root
 |-- bill_id: string (nullable = true)
 |-- patient_id: string (nullable = true)
 |-- treatment_id: string (nullable = true)
 |-- bill_date: date (nullable = true)
 |-- amount: double (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- payment_status: string (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _source_file_name: string (nullable = true)
 |-- _batch_id: string (nullable = true)
 |-- _layer: string (nullable = true)
 |-- _ingestion_date: date (nullable = true)
 |-- _pipeline_version: string (nullable = true)
 |-- _source_system: string (nullable = true)
 |-- _record_hash: string (nullable = true)
 |-- _is_duplicate: boolean (nullable = true)
 |-- _raw_row_number: long (nullable = true)



In [0]:
billing_silver = (
    billing_bronze
    .dropDuplicates(["bill_id"])

    .withColumn(
        "patient_id",
        F.trim(F.col("patient_id"))
    )

    .withColumn(
        "treatment_id",
        F.trim(F.col("treatment_id"))
    )

    .withColumn(
        "payment_method",
        F.upper(F.trim(F.col("payment_method")))
    )

    .withColumn(
        "payment_status",
        F.upper(F.trim(F.col("payment_status")))
    )

    .withColumn(
        "amount_valid_flag",
        F.col("amount") > 0
    )

    .withColumn(
        "payment_success_flag",
        F.when(
            F.col("payment_status") == "PAID",
            1
        ).otherwise(0)
    )

    .withColumn(
        "pending_flag",
        F.when(
            F.col("payment_status") == "PENDING",
            1
        ).otherwise(0)
    )

    .withColumn(
        "failed_payment_flag",
        F.when(
            F.col("payment_status") == "FAILED",
            1
        ).otherwise(0)
    )

    .withColumn(
        "insurance_payment_flag",
        F.when(
            F.col("payment_method") == "INSURANCE",
            1
        ).otherwise(0)
    )

    .withColumn(
        "billing_year",
        F.year("bill_date")
    )

    .withColumn(
        "billing_month",
        F.month("bill_date")
    )
)

In [0]:
billing_silver = (
    billing_silver

    .withColumn(
        "_dq_passed",
        (
            F.col("bill_id").isNotNull()
            &
            F.col("patient_id").isNotNull()
            &
            F.col("treatment_id").isNotNull()
            &
            F.col("bill_date").isNotNull()
            &
            F.col("amount_valid_flag")
        )
    )

    .withColumn(
        "_dq_score",
        (
            F.when(F.col("bill_id").isNotNull(), 1).otherwise(0)
            +
            F.when(F.col("patient_id").isNotNull(), 1).otherwise(0)
            +
            F.when(F.col("treatment_id").isNotNull(), 1).otherwise(0)
            +
            F.when(F.col("bill_date").isNotNull(), 1).otherwise(0)
            +
            F.when(F.col("amount_valid_flag"), 1).otherwise(0)
        ) / F.lit(5.0)
    )

    .withColumn(
        "_dq_failure_reason",
        F.when(
            F.col("_dq_passed") == False,
            F.lit("Billing validation failed")
        )
    )

    .withColumn(
        "_silver_load_timestamp",
        F.current_timestamp()
    )

    .withColumn(
        "_silver_batch_id",
        F.expr("uuid()")
    )

    .withColumn(
        "_bronze_batch_id",
        F.col("_batch_id")
    )

    .withColumn(
        "_enrichment_source",
        F.lit("bronze.billing")
    )
)

In [0]:
patient_keys = (
    spark.table(
        f"{CATALOG}.silver.patients"
    )
    .select("patient_id")
    .dropDuplicates()
)

treatment_keys = (
    spark.table(
        f"{CATALOG}.silver.treatments"
    )
    .select("treatment_id")
    .dropDuplicates()
)

billing_silver = (
    billing_silver

    .join(
        patient_keys.withColumn(
            "valid_patient",
            F.lit(True)
        ),
        on="patient_id",
        how="left"
    )

    .join(
        treatment_keys.withColumn(
            "valid_treatment",
            F.lit(True)
        ),
        on="treatment_id",
        how="left"
    )

    .withColumn(
        "valid_patient",
        F.coalesce(
            F.col("valid_patient"),
            F.lit(False)
        )
    )

    .withColumn(
        "valid_treatment",
        F.coalesce(
            F.col("valid_treatment"),
            F.lit(False)
        )
    )

    .withColumn(
        "_dq_passed",
        F.col("_dq_passed")
        &
        F.col("valid_patient")
        &
        F.col("valid_treatment")
    )
)

In [0]:
(
    billing_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        f"{CATALOG}.silver.billing"
    )
)

In [0]:
print(
    "Silver billing rows:",
    spark.table(
        f"{CATALOG}.silver.billing"
    ).count()
)

display(
    spark.table(
        f"{CATALOG}.silver.billing"
    )
)

Silver billing rows: 200


treatment_id,patient_id,bill_id,bill_date,amount,payment_method,payment_status,_ingestion_timestamp,_source_file_name,_batch_id,_layer,_ingestion_date,_pipeline_version,_source_system,_record_hash,_is_duplicate,_raw_row_number,amount_valid_flag,payment_success_flag,pending_flag,failed_payment_flag,insurance_payment_flag,billing_year,billing_month,_dq_passed,_dq_score,_dq_failure_reason,_silver_load_timestamp,_silver_batch_id,_bronze_batch_id,_enrichment_source,valid_patient,valid_treatment
T046,P019,B046,2023-12-20,1526.36,CASH,PAID,2026-08-10T16:09:50.712Z,abfss://landing@tanvihealthstore2608.dfs.core.windows.net/billings/billing.csv,f85645bc-e96c-4b26-a616-4a8fc95a69fa,BRONZE,2026-08-10,1.0,billing,0f6e49a76e3e98f453816f4cf64dee5a935eb21f9d4e585f8a2cd656becd14b3,false,10,true,1,0,0,0,2023,12,true,1.0,null,2026-08-10T17:29:07.438Z,1eb045c3-0218-4ea0-9a09-f19c74a40c78,f85645bc-e96c-4b26-a616-4a8fc95a69fa,bronze.billing,true,true
T165,P031,B165,2023-04-04,4126.66,CASH,FAILED,2026-08-10T16:09:50.712Z,abfss://landing@tanvihealthstore2608.dfs.core.windows.net/billings/billing.csv,f85645bc-e96c-4b26-a616-4a8fc95a69fa,BRONZE,2026-08-10,1.0,billing,55eec32aacec208ad3c940689a3a904fd644480e5c2460c56c55be9af04ad00c,false,55,true,0,0,1,0,2023,4,true,1.0,null,2026-08-10T17:29:07.438Z,a13c395a-ae0e-49c0-a265-4b32145b4619,f85645bc-e96c-4b26-a616-4a8fc95a69fa,bronze.billing,true,true
T100,P029,B100,2023-03-02,1551.7,CREDIT CARD,FAILED,2026-08-10T16:09:50.712Z,abfss://landing@tanvihealthstore2608.dfs.core.windows.net/billings/billing.csv,f85645bc-e96c-4b26-a616-4a8fc95a69fa,BRONZE,2026-08-10,1.0,billing,67dbc88471618097565bf23f6e8ec0db0d11c9c451a0f9dd8652154a6fe43dd1,false,68,true,0,0,1,0,2023,3,true,1.0,null,2026-08-10T17:29:07.438Z,aaeae7e8-de25-41ea-9c0b-e7ddeaaf2f07,f85645bc-e96c-4b26-a616-4a8fc95a69fa,bronze.billing,true,true
T168,P023,B168,2023-09-29,864.14,CREDIT CARD,FAILED,2026-08-10T16:09:50.712Z,abfss://landing@tanvihealthstore2608.dfs.core.windows.net/billings/billing.csv,f85645bc-e96c-4b26-a616-4a8fc95a69fa,BRONZE,2026-08-10,1.0,billing,6f4414c54030add5326967c5cd2aadf66e7d4ca92d002d5f32d48a31db788de5,false,74,true,0,0,1,0,2023,9,true,1.0,null,2026-08-10T17:29:07.438Z,3f8efffb-90e7-4ea1-b493-bcd7a9d30262,f85645bc-e96c-4b26-a616-4a8fc95a69fa,bronze.billing,true,true
T073,P040,B073,2023-12-24,2259.08,CREDIT CARD,FAILED,2026-08-10T16:09:50.712Z,abfss://landing@tanvihealthstore2608.dfs.core.windows.net/billings/billing.csv,f85645bc-e96c-4b26-a616-4a8fc95a69fa,BRONZE,2026-08-10,1.0,billing,90a11674c159be581e007b083922aa9898781d0918bbddca04f61e0c3bd5889f,false,95,true,0,0,1,0,2023,12,true,1.0,null,2026-08-10T17:29:07.438Z,1afd0d30-9348-452a-9b16-a3117716ea58,f85645bc-e96c-4b26-a616-4a8fc95a69fa,bronze.billing,true,true
T199,P017,B199,2023-05-01,1472.17,CREDIT CARD,PAID,2026-08-10T16:09:50.712Z,abfss://landing@tanvihealthstore2608.dfs.core.windows.net/billings/billing.csv,f85645bc-e96c-4b26-a616-4a8fc95a69fa,BRONZE,2026-08-10,1.0,billing,94620e6210322987718fd263eae7da4a4733201992f2ee68a9efdb672f746691,false,98,true,1,0,0,0,2023,5,true,1.0,null,2026-08-10T17:29:07.438Z,8e7cfbc3-cd56-46c4-8c9f-4390646f757a,f85645bc-e96c-4b26-a616-4a8fc95a69fa,bronze.billing,true,true
T198,P022,B198,2023-05-15,3383.72,CASH,FAILED,2026-08-10T16:09:50.712Z,abfss://landing@tanvihealthstore2608.dfs.core.windows.net/billings/billing.csv,f85645bc-e96c-4b26-a616-4a8fc95a69fa,BRONZE,2026-08-10,1.0,billing,a732ca4944000dcb258df1e3ba9bdebb5dc3537ff436e5959f5b45aa82b737f9,false,120,true,0,0,1,0,2023,5,true,1.0,null,2026-08-10T17:29:07.438Z,a8bebf3d-4696-44ef-ae23-b16da4f72736,f85645bc-e96c-4b26-a616-4a8fc95a69fa,bronze.billing,true,true
T150,P047,B150,2023-08-16,2286.42,CREDIT CARD,PAID,2026-08-10T16:09:50.712Z,abfss://landing@tanvihealthstore2608.dfs.core.windows.net/billings/billing.csv,f85645bc-e96c-4b26-a616-4a8fc95a69fa,BRONZE,2026-08-10,1.0,billing,ca4c334285e398533aaf9ce0e7dc79caa8d77721547bc102ecd416d9b202ac0d,false,152,true,1,0,0,0,2023,8,true,1.0,null,2026-0

In [0]:
(
    spark.table(
        f"{CATALOG}.silver.billing"
    )
    .groupBy("payment_status")
    .agg(
        F.count("*").alias("bill_count"),
        F.round(
            F.sum("amount"),
            2
        ).alias("total_amount")
    )
    .orderBy(
        F.col("total_amount").desc()
    )
    .show()
)

+--------------+----------+------------+
|payment_status|bill_count|total_amount|
+--------------+----------+------------+
|        FAILED|        67|   193212.94|
|       PENDING|        69|   184612.01|
|          PAID|        64|    173424.9|
+--------------+----------+------------+

